# LeetCode 267 — Palindrome Permutation II bằng Python

Link đề: https://leetcode.com/problems/palindrome-permutation-ii/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Backtracking

Skeleton LeetCode:

```python
class Solution:
    def generatePalindromes(self, s: str) -> List[str]:
        ...
```

Ghi chú: đây là bài **Premium** của LeetCode; đề bài viết theo hiểu biết về bài (và bản mô tả đã tải), nên đối chiếu lại nếu có tài khoản Premium. Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force độc lập (sinh mọi hoán vị của chuỗi rồi lọc các chuỗi đối xứng).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho chuỗi `s`, trả về **mọi hoán vị là chuỗi đối xứng (palindrome), không trùng lặp**; trả danh sách rỗng nếu không có hoán vị nào như vậy (thứ tự tùy ý).
2. Biết **điều kiện cần và đủ** để một chuỗi có hoán vị đối xứng: **nhiều nhất một ký tự có số lần xuất hiện lẻ** (đó là ký tự đặt ở giữa).
3. Nhận ra **bước đột phá**: một chuỗi đối xứng được quyết định hoàn toàn bởi **nửa đầu** của nó; ta chỉ cần liệt kê các hoán vị **khác nhau** của "nửa đầu" (mỗi ký tự lấy `số_lần // 2` bản) rồi **phản chiếu** qua ký tự giữa.
4. Đi từ **sinh mọi hoán vị rồi lọc (brute force)** → **quay lui xây từ ngoài vào trong** (cách tham khảo) → **quay lui chỉ nửa đầu rồi phản chiếu** → **`itertools.permutations` của nửa đầu + `set`** → **hoán vị kế tiếp theo thứ tự từ điển (không đệ quy)**.
5. Hiểu **mẫu quay lui** (chọn → khám phá → hoàn tác) và cách **tránh sinh trùng** bằng bộ đếm `Counter` thay vì `set` sau khi sinh.
6. Tránh các lỗi kinh điển: quên kiểm tra điều kiện số lần lẻ, quên ký tự giữa, sinh trùng khi ký tự lặp, và không hoàn tác bộ đếm khi quay lui.

## 2. Đề bài & ví dụ

Cho chuỗi `s`, trả về **tất cả các hoán vị đối xứng (palindromic permutations) không trùng lặp** của nó. Có thể trả về theo thứ tự bất kỳ. Nếu `s` không có hoán vị đối xứng nào, trả về danh sách rỗng.

**Ví dụ 1**

- Input: `s = "aabb"`
- Output: `["abba","baab"]`

**Ví dụ 2**

- Input: `s = "abc"`
- Output: `[]`

**Ràng buộc:** `1 <= s.length <= 16`; `s` chỉ gồm chữ cái thường.

In [1]:
from typing import List
import random
from collections import Counter
from itertools import permutations
from math import factorial, prod

examples = [
    ("aabb", ["abba", "baab"]),
    ("abc", []),
]
for s, expected in examples:
    print(f"s={s!r}, kỳ vọng (sắp) {sorted(expected)}")

s='aabb', kỳ vọng (sắp) ['abba', 'baab']
s='abc', kỳ vọng (sắp) []


## 3. Kiến thức nền

### 3.1. Chuỗi đối xứng và đảo ngược chuỗi `s[::-1]`

Chuỗi đối xứng (palindrome) đọc xuôi và ngược như nhau: `"abba"`, `"aba"`. Kiểm tra bằng `s == s[::-1]` (`[::-1]` là cắt lát với bước `-1`, cho chuỗi đảo ngược).

In [2]:
for s in ["abba", "aba", "abc", "a", ""]:
    print(repr(s), "->", s == s[::-1])

'abba' -> True
'aba' -> True
'abc' -> False
'a' -> True
'' -> True


### 3.2. `Counter` — đếm số lần xuất hiện của từng ký tự

`Counter(s)` cho từ điển `ký tự → số lần`. Duyệt `.items()` để lấy cặp. Chẵn/lẻ kiểm tra bằng `% 2` hoặc `& 1`.

In [3]:
cnt = Counter("aabbc")
print(cnt)                                              # Counter({'a': 2, 'b': 2, 'c': 1})
print({c: v for c, v in cnt.items() if v % 2 == 1})     # các ký tự có số lần lẻ: {'c': 1}
print(5 & 1, 4 & 1)                                     # 1 0

Counter({'a': 2, 'b': 2, 'c': 1})
{'c': 1}
1 0


### 3.3. Điều kiện để có hoán vị đối xứng

Trong một chuỗi đối xứng, mỗi ký tự (trừ ký tự ở chính giữa nếu độ dài lẻ) xuất hiện thành **cặp đối xứng**. Vậy:

- độ dài **chẵn**: mọi ký tự có số lần **chẵn**;
- độ dài **lẻ**: **đúng một** ký tự có số lần lẻ (đứng giữa), các ký tự khác chẵn.

Gộp lại: **số ký tự có số lần lẻ ≤ 1** (độ dài chẵn thì số này tự động bằng 0, độ dài lẻ thì bằng 1).

In [4]:
def can_form_palindrome(s):
    return sum(v & 1 for v in Counter(s).values()) <= 1


for s in ["aabb", "abc", "aab", "aabbc", "aabbcd", "a"]:
    print(repr(s), "->", can_form_palindrome(s))

'aabb' -> True
'abc' -> False
'aab' -> True
'aabbc' -> True
'aabbcd' -> False
'a' -> True


### 3.4. Hoán vị và việc trùng lặp

`itertools.permutations(iterable)` sinh mọi hoán vị (theo **vị trí**, nên với ký tự lặp lại ta thu được nhiều hoán vị cho cùng một chuỗi). Số hoán vị của `n` phần tử là `n!`, nhưng số chuỗi **khác nhau** của đa tập ký tự là `n! / Π(c!)` (`c` là số lần mỗi ký tự).

In [5]:
perms = list(permutations("aab"))
print(len(perms), perms[:3])                   # 6 hoán vị theo vị trí
strings = {"".join(p) for p in perms}
print(len(strings), sorted(strings))           # 3 chuỗi khác nhau: aab, aba, baa
print(factorial(3) // (factorial(2) * factorial(1)))       # 3 = 3! / (2! 1!)

6 [('a', 'a', 'b'), ('a', 'b', 'a'), ('a', 'a', 'b')]
3 ['aab', 'aba', 'baa']
3


### 3.5. Mẫu quay lui (backtracking): chọn, khám phá, hoàn tác

Quay lui xây lời giải từng bước: **chọn** một lựa chọn (và cập nhật trạng thái), **gọi đệ quy** để xây tiếp, rồi **hoàn tác** trạng thái để thử lựa chọn khác. Dùng bộ đếm `cnt` để biết còn dùng được ký tự nào, và chỉ duyệt các ký tự **khác nhau** ở mỗi bước thì không sinh trùng.

In [6]:
def sinh(cnt, path, out, total):
    if len(path) == total:
        out.append("".join(path))
        return
    for c in sorted(cnt):                 # mỗi ký tự khác nhau thử đúng một lần ở bước này
        if cnt[c] > 0:
            cnt[c] -= 1                   # chọn
            path.append(c)
            sinh(cnt, path, out, total)   # khám phá
            path.pop()                    # hoàn tác
            cnt[c] += 1


out = []
sinh(Counter("aab"), [], out, 3)
print(out)                                # ['aab', 'aba', 'baa']: không trùng

['aab', 'aba', 'baa']


### 3.5.b. Hoán vị kế tiếp theo thứ tự từ điển (next permutation)

Với danh sách đã sắp tăng, ta có thể sinh mọi hoán vị **khác nhau** theo thứ tự từ điển mà không dùng đệ quy: tìm `i` lớn nhất với `a[i] < a[i + 1]`; tìm `j` lớn nhất với `a[j] > a[i]`; đổi chỗ; đảo ngược đoạn sau `i`. Nếu không có `i` thì đã là hoán vị cuối.

In [7]:
def next_permutation(a):
    """Biến a thành hoán vị kế tiếp theo thứ tự từ điển; trả False nếu a đã là hoán vị lớn nhất."""
    i = len(a) - 2
    while i >= 0 and a[i] >= a[i + 1]:
        i -= 1
    if i < 0:
        return False
    j = len(a) - 1
    while a[j] <= a[i]:
        j -= 1
    a[i], a[j] = a[j], a[i]
    a[i + 1:] = reversed(a[i + 1:])
    return True


a = list("aab")
seen = ["".join(a)]
while next_permutation(a):
    seen.append("".join(a))
print(seen)                                # ['aab', 'aba', 'baa']

['aab', 'aba', 'baa']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ `s = "aabb"`: số lần `a: 2`, `b: 2`, không ký tự nào lẻ → có hoán vị đối xứng. Chuỗi đối xứng độ dài 4 có dạng `x y y x`: ký tự thứ 1 và thứ 4 bằng nhau, thứ 2 và thứ 3 bằng nhau. Mỗi ký tự dùng **một cặp**, nên **nửa đầu** gồm một `a` và một `b`: các nửa đầu có thể là `"ab"` hoặc `"ba"`:

| nửa đầu | ký tự giữa | phản chiếu | kết quả |
|---|---|---|---|
| `ab` | (không có) | `ab` + `ba` | `abba` |
| `ba` | (không có) | `ba` + `ab` | `baab` |

Với `s = "aabbc"`: `c` lẻ nên là ký tự giữa; nửa đầu dùng `a`, `b` (mỗi ký tự `số_lần // 2 = 1` lần): `"ab"` → `ab c ba = "abcba"`, `"ba"` → `"bacab"`.

Với `s = "abc"`: ba ký tự đều lẻ: không có hoán vị đối xứng: `[]`.

In [8]:
for s in ["aabb", "aabbc", "abc"]:
    cnt = Counter(s)
    odd = [c for c, v in cnt.items() if v & 1]
    half = "".join(c * (v // 2) for c, v in sorted(cnt.items()))
    print(f"s={s!r}: số ký tự lẻ = {len(odd)}", "-> không có" if len(odd) > 1 else f"-> giữa={odd[0] if odd else ''!r}, nửa đầu (đa tập) = {half!r}")

s='aabb': số ký tự lẻ = 0 -> giữa='', nửa đầu (đa tập) = 'ab'
s='aabbc': số ký tự lẻ = 1 -> giữa='c', nửa đầu (đa tập) = 'ab'
s='abc': số ký tự lẻ = 3 -> không có


## 5. Cách 1 — Brute Force / baseline: sinh mọi hoán vị rồi lọc

**Ý tưởng.** Dùng `permutations(s)` sinh mọi hoán vị (kể cả trùng), gom vào `set` để bỏ trùng, giữ lại các chuỗi đối xứng. Đúng theo nghĩa đen đề, nhưng số hoán vị là `n!`.

In [9]:
def palindromes_bruteforce(s: str) -> list[str]:
    result = set()
    for p in permutations(s):
        t = "".join(p)
        if t == t[::-1]:                      # đối xứng?
            result.add(t)                     # set bỏ trùng
    return list(result)


for s, expected in examples:
    print(f"{s!r} -> {sorted(palindromes_bruteforce(s))} (kỳ vọng {sorted(expected)})")

'aabb' -> ['abba', 'baab'] (kỳ vọng ['abba', 'baab'])
'abc' -> [] (kỳ vọng [])


**Độ phức tạp (Cách 1).** Gọi `n = len(s)`. Sinh `n!` hoán vị, mỗi cái kiểm tra `O(n)`: **Time `O(n · n!)`**, **Space `O(số kết quả · n)`** cho `set`. Với `n = 16` là `16! ≈ 2·10¹³`: **không khả thi**; chỉ dùng cho `n ≤ 8` làm chuẩn đối chiếu.

**Nút thắt.** Ta sinh rất nhiều hoán vị **không** đối xứng rồi bỏ đi. Một chuỗi đối xứng chỉ cần quyết định **nửa đầu**.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1: kiểm tra điều kiện.** Đếm số lần xuất hiện; nếu có nhiều hơn một ký tự có số lần lẻ thì trả `[]`.

**Bước 2: nửa đầu và ký tự giữa.** Nếu có đúng một ký tự lẻ, đó là `mid`; nếu không `mid = ""`. Mỗi ký tự `c` đóng góp `cnt[c] // 2` bản vào **đa tập nửa đầu** `half` (độ dài `n // 2`).

**Bước 3: liệt kê các hoán vị khác nhau của `half`.** Mỗi hoán vị `h` cho đúng một chuỗi đối xứng `h + mid + h[::-1]`. Hai hoán vị khác nhau của `half` cho hai chuỗi khác nhau (nửa đầu khác nhau) và mọi chuỗi đối xứng đều xuất hiện như vậy.

**Số kết quả:** `(n // 2)! / Π (cnt[c] // 2)!` (số hoán vị khác nhau của đa tập `half`). Với `n = 16` và mọi ký tự lặp đôi, tối đa là `8! = 40320`: nhỏ.

In [10]:
def build_half(s):
    cnt = Counter(s)
    odd = [c for c, v in cnt.items() if v & 1]
    if len(odd) > 1:
        return None
    mid = odd[0] if odd else ""
    half = "".join(c * (v // 2) for c, v in sorted(cnt.items()))
    return half, mid


for s in ["aabb", "aabbc", "aaabbbcc", "abc"]:
    r = build_half(s)
    if r is None:
        print(f"{s!r}: không có hoán vị đối xứng")
    else:
        half, mid = r
        count = factorial(len(half)) // prod(factorial(v) for v in Counter(half).values())
        print(f"{s!r}: nửa đầu (đa tập) = {half!r}, giữa = {mid!r}, số kết quả = {count}")

'aabb': nửa đầu (đa tập) = 'ab', giữa = '', số kết quả = 2
'aabbc': nửa đầu (đa tập) = 'ab', giữa = 'c', số kết quả = 2
'aaabbbcc': không có hoán vị đối xứng
'abc': không có hoán vị đối xứng


## 7. Các cách tiếp theo

### Cách 2 — Quay lui xây từ ngoài vào trong (cách của lời giải tham khảo)

**Giải thích.** Xác định `mid` như trên (trừ một bản của ký tự lẻ khỏi `cnt`). Bắt đầu `t = mid`; hàm `dfs(t)` chọn một ký tự `c` còn **ít nhất 2 bản**, bọc hai đầu: `t' = c + t + c`, trừ `cnt[c]` đi 2, đệ quy, rồi hoàn tác. Dừng khi `len(t) == len(s)`.

In [11]:
def palindromes_outside_in(s: str) -> list[str]:
    cnt = Counter(s)
    mid = ""
    for c, v in cnt.items():
        if v & 1:
            if mid:                                  # đã có một ký tự lẻ rồi: không thể
                return []
            mid = c
            cnt[c] -= 1                              # lấy một bản làm ký tự giữa
    ans = []

    def dfs(t):
        if len(t) == len(s):
            ans.append(t)
            return
        for c, v in cnt.items():                     # mỗi ký tự khác nhau thử một lần
            if v > 1:                                # còn đủ một cặp
                cnt[c] -= 2
                dfs(c + t + c)                       # bọc hai đầu
                cnt[c] += 2                          # hoàn tác

    dfs(mid)
    return ans


for s, expected in examples:
    print(f"{s!r} -> {sorted(palindromes_outside_in(s))} (kỳ vọng {sorted(expected)})")

'aabb' -> ['abba', 'baab'] (kỳ vọng ['abba', 'baab'])
'abc' -> [] (kỳ vọng [])


In [12]:
# Code nộp LeetCode cho Cách 2
from collections import Counter
from typing import List


class Solution:
    def generatePalindromes(self, s: str) -> List[str]:
        def dfs(t):
            if len(t) == len(s):
                ans.append(t)
                return
            for c, v in cnt.items():
                if v > 1:
                    cnt[c] -= 2
                    dfs(c + t + c)
                    cnt[c] += 2

        cnt = Counter(s)
        mid = ''
        for c, v in cnt.items():
            if v & 1:
                if mid:
                    return []
                mid = c
                cnt[c] -= 1
        ans = []
        dfs(mid)
        return ans

**Độ phức tạp Cách 2:** số kết quả `R = (n/2)! / Π(cnt//2)!`; mỗi kết quả được tạo sau `n/2` lời gọi đệ quy, mỗi cấp thử tối đa `d` ký tự (`d ≤ 26`): **Time `O(R · n)`** (cộng chi phí rẽ nhánh không đáng kể), **Space `O(n)`** cho đệ quy (không kể kết quả).

### Cách 3 — Quay lui chỉ nửa đầu rồi phản chiếu

**Giải thích.** Xây `half` từng ký tự từ trái sang phải bằng bộ đếm `half_cnt` (số bản `cnt[c] // 2`); khi đủ độ dài `n // 2` thì ghép `half + mid + half[::-1]`. Giống mẫu ở mục 3.5, chỉ khác là sinh nửa và phản chiếu một lần.

In [13]:
def palindromes_half_backtrack(s: str) -> list[str]:
    built = build_half(s)
    if built is None:
        return []
    half, mid = built
    half_cnt = Counter(half)
    size = len(half)
    ans, path = [], []

    def dfs():
        if len(path) == size:
            h = "".join(path)
            ans.append(h + mid + h[::-1])           # phản chiếu qua ký tự giữa
            return
        for c in sorted(half_cnt):
            if half_cnt[c] > 0:
                half_cnt[c] -= 1                    # chọn
                path.append(c)
                dfs()                                # khám phá
                path.pop()                           # hoàn tác
                half_cnt[c] += 1

    dfs()
    return ans


for s, expected in examples:
    print(f"{s!r} -> {sorted(palindromes_half_backtrack(s))} (kỳ vọng {sorted(expected)})")

'aabb' -> ['abba', 'baab'] (kỳ vọng ['abba', 'baab'])
'abc' -> [] (kỳ vọng [])


**Độ phức tạp Cách 3:** **Time `O(R · n)`**, **Space `O(n)`**; mỗi kết quả sinh đúng một lần nên không cần `set`.

### Cách 4 — `itertools.permutations` trên nửa đầu + `set`

**Giải thích.** Ngắn gọn: `{"".join(p) for p in permutations(half)}` cho mọi nửa đầu khác nhau (dùng `set` để bỏ trùng do ký tự lặp), rồi phản chiếu. Số hoán vị theo vị trí là `(n/2)!` (tối đa `8! = 40320`), chấp nhận được với `n ≤ 16`.

In [14]:
def palindromes_permutations_half(s: str) -> list[str]:
    built = build_half(s)
    if built is None:
        return []
    half, mid = built
    halves = {"".join(p) for p in permutations(half)}       # set bỏ các nửa trùng nhau
    return [h + mid + h[::-1] for h in halves]


for s, expected in examples:
    print(f"{s!r} -> {sorted(palindromes_permutations_half(s))} (kỳ vọng {sorted(expected)})")

'aabb' -> ['abba', 'baab'] (kỳ vọng ['abba', 'baab'])
'abc' -> [] (kỳ vọng [])


**Độ phức tạp Cách 4:** **Time `O((n/2)! · n)`** (số hoán vị theo vị trí, tối đa `8!` với `n = 16`), **Space `O((n/2)! · n)`** cho `set`. Nhanh với `n ≤ 16` nhưng lãng phí khi ký tự lặp nhiều.

### Cách 5 — Hoán vị kế tiếp theo thứ tự từ điển (không đệ quy, không `set`)

**Giải thích.** Sắp `half` tăng dần thành danh sách `a`; lặp: ghi `a` ra, rồi gọi `next_permutation(a)` cho tới khi hết. Cách này sinh đúng các hoán vị khác nhau, theo thứ tự từ điển, không trùng và không đệ quy.

In [15]:
def palindromes_next_permutation(s: str) -> list[str]:
    built = build_half(s)
    if built is None:
        return []
    half, mid = built
    a = sorted(half)
    ans = []
    while True:
        h = "".join(a)
        ans.append(h + mid + h[::-1])
        if not next_permutation(a):
            break
    return ans


for s, expected in examples:
    print(f"{s!r} -> {sorted(palindromes_next_permutation(s))} (kỳ vọng {sorted(expected)})")

'aabb' -> ['abba', 'baab'] (kỳ vọng ['abba', 'baab'])
'abc' -> [] (kỳ vọng [])


**Độ phức tạp Cách 5:** mỗi bước `next_permutation` tốn `O(n)`; **Time `O(R · n)`**, **Space `O(n)`** (không kể kết quả). Kết quả tự động theo thứ tự từ điển.

> Cách 2 là cách tham khảo; Cách 3 và 5 rõ ràng và không trùng; Cách 4 ngắn nhất; Cách 1 là chuẩn đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Cách 2 (quay lui ngoài vào) trên `s = "aabbc"` và Cách 5 trên `half = "aab"`.

In [16]:
def trace_outside_in(s):
    cnt = Counter(s)
    mid = ""
    for c, v in cnt.items():
        if v & 1:
            mid = c
            cnt[c] -= 1
    print(f"s={s!r}: giữa={mid!r}, cnt còn lại={dict(cnt)}")
    ans = []

    def dfs(t, depth=0):
        pad = "  " * depth
        if len(t) == len(s):
            print(f"{pad}t={t!r}: đủ độ dài -> THU {t!r}")
            ans.append(t)
            return
        print(f"{pad}t={t!r}, cnt={dict(cnt)}")
        for c, v in cnt.items():
            if v > 1:
                cnt[c] -= 2
                print(f"{pad}  chọn {c!r}: bọc hai đầu -> {c + t + c!r}")
                dfs(c + t + c, depth + 2)
                cnt[c] += 2

    dfs(mid)
    return ans


assert sorted(trace_outside_in("aabbc")) == ["abcba", "bacab"]

s='aabbc': giữa='c', cnt còn lại={'a': 2, 'b': 2, 'c': 0}
t='c', cnt={'a': 2, 'b': 2, 'c': 0}
  chọn 'a': bọc hai đầu -> 'aca'
    t='aca', cnt={'a': 0, 'b': 2, 'c': 0}
      chọn 'b': bọc hai đầu -> 'bacab'
        t='bacab': đủ độ dài -> THU 'bacab'
  chọn 'b': bọc hai đầu -> 'bcb'
    t='bcb', cnt={'a': 2, 'b': 0, 'c': 0}
      chọn 'a': bọc hai đầu -> 'abcba'
        t='abcba': đủ độ dài -> THU 'abcba'


In [17]:
def trace_next_perm(half):
    a = sorted(half)
    print(f"nửa đầu đa tập = {half!r}")
    step = 0
    while True:
        step += 1
        print(f"  hoán vị {step}: {''.join(a)}")
        if not next_permutation(a):
            print("  hết hoán vị")
            break


trace_next_perm("aab")

nửa đầu đa tập = 'aab'
  hoán vị 1: aab
  hoán vị 2: aba
  hoán vị 3: baa
  hết hoán vị


Bảng tóm tắt (Cách 2, `s = "aabbc"`, giữa `c`, `cnt = {a: 2, b: 2, c: 0}`):

| bước | `t` | chọn | `t` mới |
|---|---|---|---|
| gốc | `c` | `a` (còn ≥ 2) | `aca` |
| | `aca` | `b` | `bacab` (đủ 5, thu) |
| gốc | `c` | `b` | `bcb` |
| | `bcb` | `a` | `abcba` (đủ 5, thu) |

Hai kết quả: `bacab`, `abcba`. Mỗi cặp được trừ khỏi `cnt` khi chọn và cộng lại khi quay lui.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (quay lui ngoài vào). Đúng class `Solution`, đúng method `generatePalindromes(self, s: str) -> List[str]`.

In [18]:
from collections import Counter
from typing import List


class Solution:
    def generatePalindromes(self, s: str) -> List[str]:
        def dfs(t):
            if len(t) == len(s):                     # đủ độ dài: một hoán vị đối xứng hoàn chỉnh
                ans.append(t)
                return
            for c, v in cnt.items():                 # mỗi ký tự khác nhau thử đúng một lần ở mỗi cấp
                if v > 1:                            # còn ít nhất một cặp để bọc hai đầu
                    cnt[c] -= 2
                    dfs(c + t + c)
                    cnt[c] += 2                      # hoàn tác

        cnt = Counter(s)
        mid = ""
        for c, v in cnt.items():
            if v & 1:                                # ký tự có số lần lẻ
                if mid:                              # đã có một ký tự lẻ: không thể đối xứng
                    return []
                mid = c
                cnt[c] -= 1                          # một bản làm ký tự giữa
        ans = []
        dfs(mid)
        return ans


for s, expected in examples:
    assert sorted(Solution().generatePalindromes(s)) == sorted(expected)
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Điều kiện đối xứng:** nếu `t` đối xứng thì vị trí `i` và `n − 1 − i` cùng ký tự; các vị trí ghép thành cặp (trừ vị trí giữa khi `n` lẻ). Mỗi ký tự xuất hiện chẵn lần nếu không ở giữa; vậy số ký tự có số lần lẻ ≤ 1 (đúng 1 khi `n` lẻ). Ngược lại nếu điều kiện thỏa, ta tạo được chuỗi đối xứng (xếp các cặp đối xứng quanh ký tự giữa).
- **Song ánh nửa đầu ↔ chuỗi đối xứng:** một chuỗi đối xứng được xác định bởi nửa đầu `h` (độ dài `n // 2`) và ký tự giữa `mid` (cố định theo điều kiện); `t = h + mid + h[::-1]`. Đa tập ký tự của `h` phải đúng là `half` (mỗi ký tự `cnt[c] // 2` lần) để tổng số lần mỗi ký tự khớp `s`. Hai nửa đầu khác nhau cho hai chuỗi khác nhau.
- **Không trùng:** mỗi cấp đệ quy duyệt các ký tự **khác nhau** (khóa của `cnt`), nên hai nhánh khác nhau chọn ký tự khác nhau ở một bước nào đó và cho hai chuỗi khác nhau. Không cần `set`.
- **Đủ:** mọi nửa đầu hợp lệ (mọi cách sắp đa tập `half`) được đệ quy sinh ra vì ở mỗi bước ta thử mọi ký tự còn lại.
- **Hoàn tác:** `cnt[c] += 2` sau lời gọi đệ quy khôi phục trạng thái để các nhánh khác dùng đúng số bản còn lại.
- **Cách 2 (ngoài vào)** sinh cùng tập vì mỗi lần bọc ghép một cặp `c…c` quanh chuỗi giữa; chuỗi cuối cùng là chuỗi đối xứng có nửa đầu là dãy các `c` đã chọn (từ ngoài vào trong).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`, `R = (n/2)! / Π(cnt[c]/2)!` là số chuỗi đối xứng khác nhau.

- **Cách 1 (mọi hoán vị + lọc):** Time `O(n · n!)`, Space `O(R · n)`.
- **Cách 2 (quay lui ngoài vào):** Time `O(R · n)`, Space `O(n)` ngăn xếp (không kể kết quả).
- **Cách 3 (quay lui nửa đầu):** Time `O(R · n)`, Space `O(n)`.
- **Cách 4 (`permutations` nửa đầu + `set`):** Time `O((n/2)! · n)`, Space `O((n/2)! · n)`.
- **Cách 5 (next permutation):** Time `O(R · n)`, Space `O(n)`.
- Với `n ≤ 16`, `R ≤ 8! = 40320`, mỗi chuỗi dài ≤ 16: tối đa vài triệu thao tác. Kết quả cũng có kích thước `O(R · n)` nên không thể nhanh hơn.

## 12. Bộ test

Ta gom các cách (trừ brute force `n!` chỉ dùng cho `n ≤ 8`) vào `impls`. `oracle(s)` dựa vào `permutations(s)` + lọc đối xứng; với `n` lớn hơn ta kiểm tra bằng **tính chất**: mỗi phần tử là chuỗi đối xứng, cùng đa tập ký tự với `s`, đôi một khác nhau, và số phần tử đúng bằng công thức `R`.

In [19]:
sol = Solution()
impls = {
    "outside_in": palindromes_outside_in,
    "half_backtrack": palindromes_half_backtrack,
    "permutations_half": palindromes_permutations_half,
    "next_permutation": palindromes_next_permutation,
    "submit": sol.generatePalindromes,
}


def oracle(s):
    return sorted({"".join(p) for p in permutations(s) if "".join(p) == "".join(p)[::-1]})


def expected_count(s):
    cnt = Counter(s)
    if sum(v & 1 for v in cnt.values()) > 1:
        return 0
    return factorial(len(s) // 2) // prod(factorial(v // 2) for v in cnt.values())


def valid_output(s, out):
    return (len(out) == len(set(out)) == expected_count(s)
            and all(t == t[::-1] and sorted(t) == sorted(s) for t in out))


for s, expected in examples:
    assert oracle(s) == sorted(expected)
    assert sorted(palindromes_bruteforce(s)) == sorted(expected)
    for name, fn in impls.items():
        assert sorted(fn(s)) == sorted(expected), (name, s, fn(s), expected)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [20]:
def check_all(s, label):
    for name, fn in impls.items():
        out = fn(s)
        assert valid_output(s, out), (label, name, out)
    if len(s) <= 8:
        want = oracle(s)
        for name, fn in impls.items():
            assert sorted(fn(s)) == want, (label, name)
        assert sorted(palindromes_bruteforce(s)) == want
    print("PASS", label, "->", expected_count(s), "kết quả")


check_all("a", "một ký tự")
check_all("aa", "hai ký tự giống nhau")
check_all("ab", "hai ký tự khác nhau (không có)")
check_all("aab", "độ dài lẻ: giữa là b")
check_all("aabb", "ví dụ 1 của đề")
check_all("abc", "ví dụ 2 của đề")
check_all("aaaa", "một ký tự lặp: chỉ một kết quả")
check_all("aabbc", "một ký tự lẻ ở giữa")
check_all("aabbcd", "hai ký tự lẻ: không có")
check_all("abcba", "đã là palindrome")
check_all("aaabbbb", "a lẻ (3 lần), b chẵn (4 lần): a ở giữa")
check_all("aabbccdd", "bốn cặp: 4!=24 kết quả")
check_all("aabbccddeeffgghh", "n = 16 (8 cặp khác nhau): 8! = 40320 kết quả")
assert expected_count("aabbccddeeffgghh") == 40320
check_all("a" * 16, "n = 16 toàn một ký tự: 1 kết quả")
check_all("abcdefghijklmnopqrstuvwxyz"[:8] * 2, "n = 16: tám ký tự mỗi ký tự hai lần")

PASS một ký tự -> 1 kết quả
PASS hai ký tự giống nhau -> 1 kết quả
PASS hai ký tự khác nhau (không có) -> 0 kết quả
PASS độ dài lẻ: giữa là b -> 1 kết quả
PASS ví dụ 1 của đề -> 2 kết quả
PASS ví dụ 2 của đề -> 0 kết quả
PASS một ký tự lặp: chỉ một kết quả -> 1 kết quả
PASS một ký tự lẻ ở giữa -> 2 kết quả
PASS hai ký tự lẻ: không có -> 0 kết quả
PASS đã là palindrome -> 2 kết quả
PASS a lẻ (3 lần), b chẵn (4 lần): a ở giữa -> 3 kết quả
PASS bốn cặp: 4!=24 kết quả -> 24 kết quả


PASS n = 16 (8 cặp khác nhau): 8! = 40320 kết quả -> 40320 kết quả
PASS n = 16 toàn một ký tự: 1 kết quả -> 1 kết quả


PASS n = 16: tám ký tự mỗi ký tự hai lần -> 40320 kết quả


### 12.2. Tính chất và phản ví dụ

In [21]:
random.seed(267)

# Tính chất 1: nếu có kết quả thì mỗi kết quả là palindrome với cùng đa tập ký tự; số kết quả = R
for _ in range(400):
    n = random.randint(1, 12)
    s = "".join(random.choice("abcd") for _ in range(n))
    out = sol.generatePalindromes(s)
    assert valid_output(s, out), s
print("PASS tính chất: palindrome, đúng đa tập, không trùng, số lượng đúng công thức")

# Tính chất 2: có kết quả khi và chỉ khi số ký tự có số lần lẻ <= 1
for _ in range(400):
    n = random.randint(1, 12)
    s = "".join(random.choice("abc") for _ in range(n))
    assert (len(sol.generatePalindromes(s)) > 0) == can_form_palindrome(s)
print("PASS tính chất: có kết quả <=> điều kiện số lần lẻ")

# Tính chất 3: thứ tự ký tự trong s không ảnh hưởng đến tập kết quả
for _ in range(100):
    n = random.randint(1, 10)
    s = "".join(random.choice("abc") for _ in range(n))
    t = list(s)
    random.shuffle(t)
    assert sorted(sol.generatePalindromes(s)) == sorted(sol.generatePalindromes("".join(t)))
print("PASS tính chất: hoán vị s không đổi tập kết quả")

# Phản ví dụ: không kiểm tra điều kiện số lần lẻ
def sai_khong_kiem_tra(s):
    cnt = Counter(s)
    half = "".join(c * (v // 2) for c, v in sorted(cnt.items()))
    mid = "".join(c for c, v in cnt.items() if v & 1)       # có thể nhiều hơn một ký tự
    return sorted({"".join(p) + mid + "".join(p)[::-1] for p in permutations(half)})

print("abc -> sai:", sai_khong_kiem_tra("abc"), "| đúng:", sol.generatePalindromes("abc"))
assert sai_khong_kiem_tra("abc") != []

PASS tính chất: palindrome, đúng đa tập, không trùng, số lượng đúng công thức
PASS tính chất: có kết quả <=> điều kiện số lần lẻ
PASS tính chất: hoán vị s không đổi tập kết quả
abc -> sai: ['abc'] | đúng: []


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [22]:
random.seed(2026)
for trial in range(600):
    n = random.randint(1, 8)
    alphabet = random.choice(["ab", "abc", "abcd"])
    s = "".join(random.choice(alphabet) for _ in range(n))
    want = oracle(s)
    for name, fn in impls.items():
        assert sorted(fn(s)) == want, (name, s)
    assert sorted(palindromes_bruteforce(s)) == want

for trial in range(40):                          # n tới 16: các cách so sánh với nhau và với công thức
    n = random.randint(9, 16)
    letters = random.sample("abcdefgh", random.randint(2, 6))
    half = [random.choice(letters) for _ in range(n // 2)]
    s_list = half + half
    if n % 2:
        s_list.append(random.choice(letters))
    random.shuffle(s_list)
    s = "".join(s_list)
    results = {name: sorted(fn(s)) for name, fn in impls.items()}
    base = results["outside_in"]
    assert valid_output(s, base)
    for name, got in results.items():
        assert got == base, (name, s)
print("PASS 600 chuỗi nhỏ khớp oracle độc lập; 40 chuỗi dài (n tới 16) khớp giữa các cách và công thức")

PASS 600 chuỗi nhỏ khớp oracle độc lập; 40 chuỗi dài (n tới 16) khớp giữa các cách và công thức


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Không kiểm tra điều kiện số ký tự lẻ

In [23]:
def sai_khong_kiem_tra_dk(s):
    cnt = Counter(s)
    half = "".join(c * (v // 2) for c, v in sorted(cnt.items()))
    mid = "".join(c for c, v in cnt.items() if v & 1)       # sai: có thể gom NHIỀU ký tự lẻ vào giữa
    return sorted({"".join(p) + mid + "".join(p)[::-1] for p in permutations(half)})


print("abc -> sai:", sai_khong_kiem_tra_dk("abc"), "| đúng:", Solution().generatePalindromes("abc"))

abc -> sai: ['abc'] | đúng: []


**Vì sao sai:** nếu có từ hai ký tự lẻ trở lên thì **không** tồn tại hoán vị đối xứng. Code sai gom tất cả ký tự lẻ vào `mid` (ví dụ `"abc"`), tạo chuỗi `"abc"` không đối xứng. Phải `return []` ngay khi có ký tự lẻ thứ hai (hoặc kiểm tra `sum(v & 1) <= 1` từ đầu).

### Lỗi 2: Quên ký tự giữa

In [24]:
def sai_quen_giua(s):
    cnt = Counter(s)
    if sum(v & 1 for v in cnt.values()) > 1:
        return []
    half = "".join(c * (v // 2) for c, v in sorted(cnt.items()))
    halves = {"".join(p) for p in permutations(half)}
    return [h + h[::-1] for h in halves]               # sai: thiếu ký tự giữa


print("aabbc -> sai:", sorted(sai_quen_giua("aabbc")), "| đúng:", sorted(Solution().generatePalindromes("aabbc")))

aabbc -> sai: ['abba', 'baab'] | đúng: ['abcba', 'bacab']


**Vì sao sai:** với độ dài lẻ phải ghép `h + mid + h[::-1]`. Thiếu `mid` cho chuỗi độ dài chẵn `4` (`"abba"`), không phải hoán vị của `"aabbc"` (mất một ký tự `c`).

### Lỗi 3: Sinh trùng khi có ký tự lặp (không dùng bộ đếm / `set`)

In [25]:
def sai_trung(s):
    cnt = Counter(s)
    if sum(v & 1 for v in cnt.values()) > 1:
        return []
    half = "".join(c * (v // 2) for c, v in sorted(cnt.items()))
    mid = "".join(c for c, v in cnt.items() if v & 1)
    return ["".join(p) + mid + "".join(p)[::-1] for p in permutations(half)]      # permutations theo vị trí -> trùng


out = sai_trung("aaaabb")
print("aaaabb -> số kết quả sai:", len(out), "| phân biệt:", len(set(out)), "| đúng:", len(Solution().generatePalindromes("aaaabb")))

aaaabb -> số kết quả sai: 6 | phân biệt: 3 | đúng: 3


**Vì sao sai:** `permutations(half)` coi các ký tự giống nhau ở các vị trí khác nhau là phần tử khác nhau, nên cùng một chuỗi xuất hiện nhiều lần (`"aab"` ra `2` lần cho mỗi nửa đầu). Phải bỏ trùng bằng `set`, hoặc dùng quay lui theo bộ đếm / hoán vị kế tiếp (không sinh trùng).

### Lỗi 4: Không hoàn tác bộ đếm khi quay lui

In [26]:
def sai_khong_hoan_tac(s):
    cnt = Counter(s)
    mid = ""
    for c, v in cnt.items():
        if v & 1:
            if mid:
                return []
            mid = c
            cnt[c] -= 1
    ans = []

    def dfs(t):
        if len(t) == len(s):
            ans.append(t)
            return
        for c, v in list(cnt.items()):
            if v > 1:
                cnt[c] -= 2
                dfs(c + t + c)
                # sai: thiếu cnt[c] += 2

    dfs(mid)
    return ans


print("aabb -> sai:", sai_khong_hoan_tac("aabb"), "| đúng:", sorted(Solution().generatePalindromes("aabb")))

aabb -> sai: ['baab'] | đúng: ['abba', 'baab']


**Vì sao sai:** sau khi thử nhánh chọn `a`, bộ đếm của `a` không được trả lại; nhánh chọn `b` ở cấp trên lại thấy `a` đã hết, và ngược lại. Kết quả thiếu các hoán vị. Mẫu quay lui luôn là: **chọn → đệ quy → hoàn tác**.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "hoán vị đối xứng / palindrome permutation": đếm số lần; điều kiện **≤ 1 ký tự lẻ**; **chỉ liệt kê nửa đầu** rồi phản chiếu (giảm `n!` xuống `(n/2)!`).
- Muốn không sinh trùng khi có phần tử lặp: quay lui theo `Counter` (mỗi cấp thử mỗi **giá trị khác nhau** một lần) hoặc dùng `next_permutation` trên danh sách đã sắp.
- Mẫu quay lui: **chọn → đệ quy → hoàn tác**; trạng thái dùng chung (bộ đếm, đường đi) phải được khôi phục.
- Biết trước số kết quả (`(n/2)!/Π(c/2)!`) giúp kiểm tra bài và ước lượng thời gian.
- Bài cùng họ: Palindrome Permutation (LeetCode 266), Permutations II (47), Letter Combinations, Subsets II.
- Test: oracle "mọi hoán vị + lọc" cho `n ≤ 8`, kiểm tra **tính chất** cho `n` lớn.

## 15. Biến thể của bài

1. **Palindrome Permutation (LeetCode 266):** chỉ hỏi có hoán vị đối xứng hay không.
2. **Đếm số hoán vị đối xứng** (không liệt kê): công thức `R`.
3. **Hoán vị đối xứng nhỏ nhất / lớn nhất** theo thứ tự từ điển.
4. **Hoán vị đối xứng thứ `k`** theo thứ tự từ điển.
5. **Hoán vị đối xứng có độ dài chẵn** hoặc sinh bằng cách **chèn bớt ký tự** (Valid Palindrome II).

In [27]:
def count_palindromes(s):
    """Biến thể 2."""
    return expected_count(s)


def smallest_palindrome(s):
    """Biến thể 3: nhỏ nhất (nửa đầu sắp tăng) hoặc None."""
    built = build_half(s)
    if built is None:
        return None
    half, mid = built
    return half + mid + half[::-1]


def largest_palindrome(s):
    built = build_half(s)
    if built is None:
        return None
    half, mid = built
    h = "".join(sorted(half, reverse=True))
    return h + mid + h[::-1]


def kth_palindrome(s, k):
    """Biến thể 4: hoán vị đối xứng thứ k (đếm từ 1) theo thứ tự từ điển; None nếu k vượt quá."""
    built = build_half(s)
    if built is None:
        return None
    half, mid = built
    cnt = Counter(half)
    size = len(half)
    out = []

    def multiset_perms(cnt, remaining):
        return factorial(remaining) // prod(factorial(v) for v in cnt.values())

    if k > multiset_perms(cnt, size):
        return None
    for position in range(size):
        for c in sorted(cnt):
            if cnt[c] == 0:
                continue
            cnt[c] -= 1
            ways = multiset_perms(cnt, size - position - 1)
            if k <= ways:
                out.append(c)
                break
            k -= ways
            cnt[c] += 1
    h = "".join(out)
    return h + mid + h[::-1]


assert count_palindromes("aabb") == 2 and count_palindromes("abc") == 0
assert smallest_palindrome("aabb") == "abba" and largest_palindrome("aabb") == "baab"
assert smallest_palindrome("abc") is None
for s in ["aabb", "aabbc", "aabbccd", "aaabbcc"]:
    allp = sorted(sol.generatePalindromes(s))
    for k in range(1, len(allp) + 1):
        assert kth_palindrome(s, k) == allp[k - 1], (s, k)
    assert kth_palindrome(s, len(allp) + 1) is None
print("PASS biến thể: đếm, nhỏ nhất/lớn nhất, thứ k")

PASS biến thể: đếm, nhỏ nhất/lớn nhất, thứ k


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi hoán vị + lọc | `permutations(s)`, `set`, kiểm tra đối xứng | `O(n·n!)` | `O(R·n)` | oracle, `n ≤ 8` |
| 2. Quay lui ngoài vào | bọc hai đầu bằng một cặp | `O(R·n)` | `O(n)` | **cách nộp bài** (lời giải tham khảo) |
| 3. Quay lui nửa đầu | sinh `half` rồi phản chiếu | `O(R·n)` | `O(n)` | rõ ràng, không trùng |
| 4. `permutations` nửa đầu + `set` | ngắn gọn | `O((n/2)!·n)` | `O((n/2)!·n)` | `n ≤ 16`, ít ký tự lặp |
| 5. Next permutation | hoán vị kế tiếp theo từ điển | `O(R·n)` | `O(n)` | không đệ quy, kết quả đã sắp |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"abcba" == "abcba"[::-1]`, `can_form_palindrome("aabbc")`, `can_form_palindrome("abcd")`, `sorted(Solution().generatePalindromes("aa"))`, `Solution().generatePalindromes("ab")`.

**Bài 2 (tính tay).** Với `s = "aabbcc"`: tính nửa đầu (đa tập), số kết quả `R` bằng công thức, rồi liệt kê cả sáu chuỗi đối xứng. Làm lại với `s = "aaabb"`.

**Bài 3 (điền TODO).** Hoàn thành `pal_todo(s)` theo Cách 3 (quay lui nửa đầu).

In [28]:
def pal_todo(s):
    cnt = Counter(s)
    # TODO 1: nếu số ký tự có số lần lẻ > 1 trả []; mid = ký tự lẻ (hoặc "")
    # TODO 2: half_cnt = {c: v // 2} và size = len(s) // 2
    # TODO 3: quay lui xây path đủ size ký tự: chọn c còn > 0, giảm, đệ quy, hoàn tác
    # TODO 4: khi đủ size ghép h + mid + h[::-1] vào kết quả
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `pal_next_perm2(s)` theo Cách 5 (sắp `half` rồi gọi `next_permutation` cho tới khi hết). Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `has_palindrome_permutation(s)` (LeetCode 266). (b) Viết `lexicographically_smallest_palindrome_perm(s)` trả về hoán vị đối xứng nhỏ nhất theo thứ tự từ điển (hoặc `""` nếu không có).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `s[::-1]` đảo chuỗi; điều kiện là tối đa một ký tự có số lần lẻ. Bài 2: mỗi ký tự lấy `số_lần // 2` bản vào nửa đầu.
- **Mức 2 (vừa):** Bài 1: `True`, `True`, `False`, `['aa']`, `[]`. Bài 2: `"aabbcc"`: nửa đầu `{a, b, c}`, `R = 3! = 6`: (sáu hoán vị của `abc`: `abc→abccba`, `acb→acbbca`, `bac→baccab`, `bca→bcaacb`, `cab→cabbac`, `cba→cbaabc`). `"aaabb"`: `a:3` lẻ (giữa `a`), `b:2`; nửa đầu `{a, b}` (`a: 1`, `b: 1`): `ab + a + ba = "ababa"`, `ba + a + ab = "baaab"`. Bài 3: bốn TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `a = sorted(half)`, lặp ghi `h + mid + h[::-1]` rồi `next_permutation(a)`. Bài 5(a): `sum(v & 1 for v in Counter(s).values()) <= 1`. Bài 5(b): nửa đầu sắp tăng + `mid` + đảo ngược nửa đầu.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert "abcba" == "abcba"[::-1]
assert can_form_palindrome("aabbc") is True and can_form_palindrome("abcd") is False
assert sorted(Solution().generatePalindromes("aa")) == ["aa"]
assert Solution().generatePalindromes("ab") == []

# Bài 2
assert sorted(Solution().generatePalindromes("aabbcc")) == sorted(["abccba", "acbbca", "baccab", "bcaacb", "cabbac", "cbaabc"])
assert expected_count("aabbcc") == 6
assert sorted(Solution().generatePalindromes("aaabb")) == ["ababa", "baaab"]

# Bài 3
def pal_todo_ans(s):
    cnt = Counter(s)
    odds = [c for c, v in cnt.items() if v & 1]
    if len(odds) > 1:
        return []
    mid = odds[0] if odds else ""
    half_cnt = {c: v // 2 for c, v in cnt.items()}
    size = len(s) // 2
    ans, path = [], []

    def dfs():
        if len(path) == size:
            h = "".join(path)
            ans.append(h + mid + h[::-1])
            return
        for c in sorted(half_cnt):
            if half_cnt[c] > 0:
                half_cnt[c] -= 1
                path.append(c)
                dfs()
                path.pop()
                half_cnt[c] += 1

    dfs()
    return ans

for s, expected in examples:
    assert sorted(pal_todo_ans(s)) == sorted(expected)

# Bài 4
def pal_next_perm2(s):
    cnt = Counter(s)
    odds = [c for c, v in cnt.items() if v & 1]
    if len(odds) > 1:
        return []
    mid = odds[0] if odds else ""
    a = sorted("".join(c * (v // 2) for c, v in cnt.items()))
    ans = []
    while True:
        h = "".join(a)
        ans.append(h + mid + h[::-1])
        if not next_permutation(a):
            return ans

for s, expected in examples:
    assert sorted(pal_next_perm2(s)) == sorted(expected)

# Bài 5(a), 5(b)
def has_palindrome_permutation(s):
    return sum(v & 1 for v in Counter(s).values()) <= 1

def lexicographically_smallest_palindrome_perm(s):
    r = smallest_palindrome(s)
    return "" if r is None else r

assert has_palindrome_permutation("carerac") and not has_palindrome_permutation("code")
assert lexicographically_smallest_palindrome_perm("aabbc") == "abcba"
assert lexicographically_smallest_palindrome_perm("abc") == ""
random.seed(19)
for _ in range(200):
    s = "".join(random.choice("abc") for _ in range(random.randint(1, 8)))
    allp = oracle(s)
    assert lexicographically_smallest_palindrome_perm(s) == (allp[0] if allp else "")
    assert has_palindrome_permutation(s) == bool(allp)
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm; điều kiện ≤ 1 ký tự lẻ; quay lui bọc cặp từ giữa ra ngoài)

```python
class Solution:
    def generatePalindromes(self, s: str) -> List[str]:
        def dfs(t):
            if len(t) == len(s):
                ans.append(t)
                return
            for c, v in cnt.items():
                if v > 1:
                    cnt[c] -= 2
                    dfs(c + t + c)
                    cnt[c] += 2

        cnt = Counter(s)
        mid = ""
        for c, v in cnt.items():
            if v & 1:
                if mid:
                    return []
                mid = c
                cnt[c] -= 1
        ans = []
        dfs(mid)
        return ans
```

**Checklist**

- [ ] Kiểm tra điều kiện: **tối đa một** ký tự có số lần lẻ, nếu không trả `[]`?
- [ ] Có ký tự giữa (`mid`) khi độ dài lẻ và trừ một bản của nó khỏi bộ đếm?
- [ ] Quay lui theo **bộ đếm** (mỗi cấp thử mỗi ký tự khác nhau một lần) để **không trùng**?
- [ ] Mẫu **chọn → đệ quy → hoàn tác** (`cnt[c] += 2`)?
- [ ] Điều kiện dừng `len(t) == len(s)` (hoặc đủ nửa đầu)?
- [ ] Đã thử: `n = 1`, ký tự lặp một loại, hai ký tự lẻ, `n = 16` (`8!` kết quả)?
- [ ] Đã đối chiếu với brute force (mọi hoán vị + lọc) cho `n ≤ 8` và kiểm tra tính chất cho `n` lớn?
- [ ] Skeleton đúng `class Solution` / `def generatePalindromes(self, s: str) -> List[str]`?